# FADE · STAGE 2 — does CTC work on questions it has never seen?

400 fresh train questions, offset 1500 — where the pool-building run stopped.

**One question at a time: attempt it, judge it, cure it there and then, move on.**

```
gen 1   model solves THIS question              -> trace, answer
cpu     CTC reads (question, trace), gold-free:
            p_wrong          is this trace wrong?
            predicted_type   which of the eight failures?
gen 2   if p_wrong >= 0.683: the matched cure, solved again, right now
score   gold opened LAST, changes nothing above it
```

Then the next question. No step reads any other problem, so stage 3 is this same
script with the scoring removed.

### Gold is required here, and that is not a contradiction

You cannot say whether CTC was right without knowing whether the trace was actually
wrong and what actually went wrong. Both labels come from gold. The rule is about
**order**, not abstinence: gold is opened after the cure has already run, and nothing
it produces feeds back. The script proves this rather than claiming it — it checks
every flag reproduces from `p_wrong >= threshold` alone, and refuses to report a run
where any flag does not.

### Why 0.683 and not 0.5

A flag pays only where precision beats `damage/(recovery+damage)` = **82.6%**. But a
budget quantile needs the whole distribution, which a per-question pipeline does not
have. So `ctc_train2.py` measures the mapping out of fold and **ships it inside the
model**; 40% becomes a fixed cut one question can be judged against:

| budget | cut | precision |
|---|---|---|
| 10% | 0.90 | 97.3% |
| 20% | 0.83 | 95.7% |
| **40%** | **0.683** | **90.8%** ← default |
| 70% | 0.490 | 82.6% ← break-even |
| 100% | 0.092 | 71.2% |

A fixed 0.5 sits *below* break-even. That was the biggest bug in the previous
version of this notebook.

### The retry is the typed arm, not a lookalike

Stage 2 calls `kaggle_run.apply_cure()` — the exact function the typed arm uses, with
only the final scoring removed. Order inside: question-aware retrieval over the pool
→ type-fit re-rank → layered ordering (most relevant first, type-matched last, nearest
the question) → approach demo → instruction + failure-mode naming + negatives into the
**system** message → prefill → generate → acceptance resample → unchanged-answer
resample → symbolic repair.

Before this, stage 2 had reimplemented a *subset*: exemplars and an instruction, but
no reroute, no naming, no prefill, no resamples, no repair. It would have measured a
weaker pipeline than the one that produced 42.0%.

### What to expect — NOT the stage-1 headline

Stage 1 averages pass-1 **and** retry traces. This stage only ever reads a **first
attempt**:

| slice | n | wrong% | detect AUROC | type acc |
|---|---|---|---|---|
| **pass-1** | 1500 | 71.2% | **0.776** | **0.598** |
| retry | 1068 | 81.5% | 0.683 | 0.679 |
| both (the headline) | 2568 | 75.5% | 0.751 | 0.635 |

Detection should land **above** the headline, typing **below** it.

### Two scorecards at the end

**1 · Was the judge right?** precision, recall, AUROC, type accuracy.
**2 · Did the model get more questions right?** pass 1 vs final, McNemar.

They are independent and can move in opposite directions: a perfect judge gains
nothing if the cure cannot fix what it found, and a working cure is wasted on traces
the judge never flags.

**Sidebar:** GPU T4/P100 · Internet on · Secrets `HF_TOKEN` and `GH_TOKEN`.
**Budget:** ~2.4 GPU-h (400 solves + ~160 retries).

### 1 · Setup

In [1]:
import os, sys, shutil, subprocess, json, datetime
!pip -q install -U "transformers>=4.40" accelerate sentence-transformers sympy scikit-learn datasets scipy joblib 2>/dev/null | tail -1
REPO='/kaggle/working/fade'; STORES='/kaggle/working/stores'; os.makedirs(STORES,exist_ok=True)
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
sys.path.insert(0,REPO)
import importlib, config; importlib.reload(config)
VER=config.FADE_CODE_VERSION; assert VER=='audit-2', f'STALE CODE {VER!r}'
# ctc_train.py stays required only so an old v1 joblib can still be unpickled.
for fn in ('ctc_train.py','ctc_train2.py','ctc_data.py','ctc_features.py',
           'run_ctc_validate.py','cure_bank.py','negatives.py','autopush.py',
           'kaggle_run.py','symbolic_repair.py'):
    assert os.path.exists(f'{REPO}/{fn}'), f'{fn} missing -- push it to the repo'
# [D76] stage 2 must run the SAME cure as the typed arm.
import kaggle_run as _K
assert hasattr(_K,'apply_cure'), (
    'kaggle_run.py is the pre-D76 version -- stage 2 would run a WEAKER retry '
    '(no reroute, naming, prefill, resamples or repair). Push it first.')
from kaggle_secrets import UserSecretsClient
_us=UserSecretsClient(); HF=_us.get_secret('HF_TOKEN'); GH=_us.get_secret('GH_TOKEN')
os.environ['FADE_HF_TOKEN']=HF
MODEL='meta-llama/Llama-2-7b-chat-hf'; TAG=VER.replace('-','')
print('code',VER,'| apply_cure present | OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.9 MB/s eta 0:00:00
Cloning into '/kaggle/working/fade'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 46 (delta 0), reused 38 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 1.40 MiB | 6.19 MiB/s, done.
Torch OK
 sentence-transformers OK
code audit-2 | apply_cure present | OK


### 2 · Capability guard

`run_ctc_validate.py` must accept every flag this notebook passes, **before** a GPU
is touched. The generic notebook once lost a whole session to a missing `--store-dir`
that the version banner did not catch. `--flag-budget` and `--type-conf-gate` are new
in this revision, so an un-pushed script fails here rather than three hours in.

In [2]:
h = subprocess.run([sys.executable,'run_ctc_validate.py','--help'], cwd=REPO,
                   capture_output=True, text=True)
need = ['--ctc','--pool-store','--store-dir','--retry-flagged','--no-score',
        '--retrieval','--flag-budget','--type-conf-gate','--negatives']
missing = [k for k in need if k not in h.stdout]
print(h.stdout[:600] if h.stdout else h.stderr[:600])
if missing:
    raise SystemExit(f"\n  STOP: run_ctc_validate.py is missing {missing}. Push the "
                     "updated file to the repo and re-run from cell 1.")
print('\n  guard passed')

Torch OK
 sentence-transformers OK
usage: run_ctc_validate.py [-h] --ctc CTC --pool-store POOL_STORE --store-dir
                           STORE_DIR [--n-problems N_PROBLEMS]
                           [--eval-offset EVAL_OFFSET] [--model MODEL]
                           [--seed-pool SEED_POOL] [--retrieval {knn,3stage}]
                           [--exemplar-budget EXEMPLAR_BUDGET]
                           [--max-new-tokens MAX_NEW_TOKENS]
                           [--flag-budget FLAG_BUDGET] [--threshold THRESHOLD]
                           [--type-conf-gate TYPE_CONF_GATE] [--negative

  guard passed


In [3]:
# --- github sync (nothing conceptual) ----------------------------------
AUTH=f'https://{GH}@github.com/S2V3/fade.git'; BRANCH='results'
def sh(cmd):
    p=subprocess.Popen(cmd,shell=True,cwd=REPO,stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT,text=True,bufsize=1)
    for ln in p.stdout: print(ln,end='',flush=True)
    p.wait(); return p.returncode
def _rows(p):
    f=os.path.join(p,'results.jsonl')
    return sum(1 for _ in open(f)) if os.path.exists(f) else 0
def _ver(p):
    try: return json.load(open(os.path.join(p,'config_snapshot.json'))).get('FADE_CODE_VERSION')
    except Exception: return None
def _clone(w):
    shutil.rmtree(w,ignore_errors=True)
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode:
        subprocess.run(['git','clone','--depth','1',AUTH,w],check=True,capture_output=True,text=True)
        subprocess.run(['git','checkout','-b',BRANCH],cwd=w,capture_output=True,text=True)
def fetch_any(prefix, dest, need_file=None):
    """Restore the newest remote dir whose name starts with `prefix`."""
    w=f'/kaggle/working/_rs_{abs(hash(prefix))%9999}'; c=[]
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode==0 and os.path.isdir(f'{w}/results'):
        for d in sorted(os.listdir(f'{w}/results')):
            if d.startswith(prefix):
                full=f'{w}/results/{d}'
                if os.path.isdir(full) and (need_file is None or os.path.exists(f'{full}/{need_file}')):
                    c.append((d, full))
    if c:
        c.sort()
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(c[-1][1], dest)
        print(f'  restored {c[-1][0]} -> {dest}')
        return True
    print(f'  nothing remote matching {prefix}*')
    return False
def fetch_store(run,dest):
    w=f'/kaggle/working/_rs_{run}'; c=[]
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode==0 and os.path.isdir(f'{w}/results'):
        for d in sorted(os.listdir(f'{w}/results')):
            if d.startswith(f'store_{run}_'):
                full=f'{w}/results/{d}'
                if os.path.isdir(full) and _ver(full)==VER and _rows(full)>0: c.append((_rows(full),d,full))
    c.sort(reverse=True)
    print(f'  [{run}] local {_rows(dest)} | best remote {c[0][0] if c else 0}')
    if c and c[0][0]>_rows(dest):
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(c[0][2],dest); print(f'  RESTORED {c[0][0]}')
    return _rows(dest)
def push_dir(src,name):
    if not os.path.isdir(src): return
    stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    w='/kaggle/working/_ck'; _clone(w)
    for dest in (f'{w}/results/{name}_{stamp}', f'{w}/results/{name}_latest'):
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(src,dest)
    for c in (['git','config','user.email','fade@kaggle'],['git','config','user.name','fade'],
              ['git','add','-A'],['git','commit','-m',f'{name}_{stamp}']):
        subprocess.run(c,cwd=w,capture_output=True,text=True)
    p=subprocess.run(['git','push','-u','origin',BRANCH],cwd=w,capture_output=True,text=True)
    if p.returncode:
        subprocess.run(['git','pull','--rebase','origin',BRANCH],cwd=w,capture_output=True,text=True)
        p=subprocess.run(['git','push','-u','origin',BRANCH],cwd=w,capture_output=True,text=True)
    print(f"  push {name}: {'ok' if p.returncode==0 else 'FAILED -- ' + p.stderr.strip()[:120]}")
print('sync ready')

sync ready


### 3 · Get the CTC v2 model — retraining locally if the push failed

Stage 1 saves `ctc2.joblib`. If it is not on the branch, retraining costs ~3 minutes
of CPU and is deterministic (`random_state=0`), so it reproduces the identical model
rather than a different one.

`ctc_data.py` assembles the training set from the **latest typed run only**, and
`ctc_train2.py` evaluates with `GroupKFold` by question — the split that does not let
a question's pass-1 trace teach its own retry.

In [4]:
TRAIN_RUN=f'typed_train_{TAG}'; TRAIN=f'{STORES}/store_{TRAIN_RUN}'
CTC_DIR=f'{STORES}/ctc2_{TAG}'
n=fetch_store(TRAIN_RUN, TRAIN)
assert n>0, 'no finished typed train store on the results branch'

CTC=f'{CTC_DIR}/ctc2.joblib'
got = fetch_any(f'ctc2_{TAG}', CTC_DIR, need_file='ctc2.joblib')
if not got or not os.path.exists(CTC):
    print('\n  no CTC v2 on the branch -- retraining (CPU, ~3 min, deterministic)')
    os.makedirs(CTC_DIR, exist_ok=True)
    rc = sh(f'python ctc_data.py --stores {TRAIN} --out {CTC_DIR}/ctc_dataset.jsonl')
    assert rc==0, 'ctc_data.py failed'
    rc = sh(f'python ctc_train2.py --data {CTC_DIR}/ctc_dataset.jsonl '
            f'--out-dir {CTC_DIR}')
    assert rc==0, 'ctc_train2.py failed'
    push_dir(CTC_DIR, f'ctc2_{TAG}')
assert os.path.exists(CTC), 'still no ctc2.joblib'

rep=json.load(open(f'{CTC_DIR}/ctc2_report.json'))
print(f"\n  n {rep['n']}  wrong {rep['n_wrong']}")
print(f"  detect AUROC {rep['detect']['auroc']:.3f}"
      f"   (expect ~0.78 here -- this stage sees pass-1 traces only)")
print(f"  type   acc   {rep['type']['accuracy']:.3f}"
      f"  majority {rep['type']['majority']:.3f}"
      f"   (expect ~0.60 here, same reason)")
assert rep['detect']['auroc']>=0.60, 'detector too weak to validate'
assert rep['type']['accuracy']>rep['type']['majority'], 'type head no better than majority'
print('  ready')

  [typed_train_audit2] local 0 | best remote 2756
  RESTORED 2756
  restored ctc2_audit2_latest -> /kaggle/working/stores/ctc2_audit2

  n 2568  wrong 1938
  detect AUROC 0.758   (expect ~0.78 here -- this stage sees pass-1 traces only)
  type   acc   0.624  majority 0.216   (expect ~0.60 here, same reason)
  ready


### 4 · Run — one number controls fresh vs resume

`ATTEMPT` is part of the store name, so it decides which of the two things happens
and you never flip a mode by hand:

- **Bump `ATTEMPT`** → nothing on the branch matches → nothing restored, starts at
  problem 1.
- **Leave it alone and re-run** → the checkpoint for that name is found → resumes.

So bump it once now, then change nothing. Every later re-run resumes *this* attempt.

Each row is written complete — trace, judgement, cure and score together — so a
resume never has a half-finished problem to reconcile. `autopush.py` pushes every 10
minutes **while the run continues**, so a lost kernel costs at most 10 minutes.

In [5]:
import autopush

# ---------------------------------------------------------------- the knob
ATTEMPT = 2          # bump for a FRESH run. leave alone to RESUME that run.
# --------------------------------------------------------------------------
N=400; OFFSET=1500; RETRY_FLAGGED=True; PUSH_EVERY_MIN=10
FLAG_BUDGET=0.40     # -> p_wrong >= 0.683 via the model's shipped calibration
TYPE_GATE=0.50       # below this confidence, use the generic cure. 0 disables.
RETRIEVAL='3stage'   # must match the train run, or the traces shift distribution

RUN=f'ctcval_train_{TAG}_a{ATTEMPT}'; STORE=f'{STORES}/store_{RUN}'

# A stale LOCAL directory resumes just as surely as a remote checkpoint and is
# invisible on the branch. Clear it before asking what the branch has, so "fresh"
# means fresh in both places.
if not os.path.exists(f'{STORE}/results.jsonl'):
    shutil.rmtree(STORE, ignore_errors=True)

fetch_store(RUN, STORE); before=_rows(STORE)
if before == 0:
    print(f'\n  FRESH START  attempt {ATTEMPT} -- nothing named store_{RUN}_* on the branch.')
    print(f'  Checkpoints push under that name every {PUSH_EVERY_MIN} min.')
    print(f'  If this kernel dies, re-run with ATTEMPT={ATTEMPT} to resume.')
else:
    print(f'\n  RESUMING attempt {ATTEMPT}: {before}/{N} questions done.')
    print(f'  To start over instead, set ATTEMPT={ATTEMPT+1} and re-run this cell.')

flag = '--retry-flagged' if RETRY_FLAGGED else '--no-retry-flagged'
rc = 1
ap = autopush.start(STORE, RUN, GH, minutes=PUSH_EVERY_MIN)
try:
    rc = sh(f'python -u run_ctc_validate.py --ctc {CTC} '
            f'--pool-store {TRAIN} --store-dir {STORE} --n-problems {N} '
            f'--eval-offset {OFFSET} --flag-budget {FLAG_BUDGET} '
            f'--type-conf-gate {TYPE_GATE} {flag} '
            f'--retrieval {RETRIEVAL} --model {MODEL} --show-every 25 '
            f'--secret-name HF_TOKEN')
finally:
    ap.stop()          # stops the thread and forces a final push
print(f'\n  attempt {ATTEMPT}: {before} -> {_rows(STORE)} questions  (exit {rc})')
assert rc==0, f'run_ctc_validate.py FAILED with exit {rc} -- read the error above'
assert _rows(STORE)>0, 'no rows written -- do not interpret anything below'

  [ctcval_train_audit2_a2] local 0 | best remote 0

  FRESH START  attempt 2 -- nothing named store_ctcval_train_audit2_a2_* on the branch.
  Checkpoints push under that name every 10 min.
  If this kernel dies, re-run with ATTEMPT=2 to resume.
  [autopush ctcval_train_audit2_a2] started, every 10 min -> /kaggle/working/stores/store_ctcval_train_audit2_a2
Torch OK
 sentence-transformers OK
  generation module version: v5-hashline
  v5-hashline ACTIVE | N_SHOTS=8 rep_penalty=1.15 no_repeat_ngram=0
  ban_strings=['\\begin{code}', '\\end{code}', '```'] | stop_markers=['\nQuestion:', '\nQ:', '\nExample', '\nProblem:']
  preprocessing: normalize_traces=True
  HF token loaded from Kaggle Secret 'HF_TOKEN'
  HF identity: S2V3 (type=user)
  gated-access probe OK: meta-llama/Llama-2-7b-chat-hf reachable (16 files)

  CTC v2 loaded: detect AUROC 0.7583435713466673 | type acc 0.6238390092879257 (majority 0.21620227038183695)
  expected on pass-1 traces: detect ~0.78, type ~0.60
  pool: 361 gold-s

### If the session dies anyway

Everything up to the last checkpoint is on the branch under `store_{RUN}_*`. Re-run
cells 1–4 **without touching `ATTEMPT`**: `fetch_store` restores the largest matching
store and the runner skips the questions already done.

To force a push at any moment, from a new cell:

```python
import autopush
autopush.AutoPush(STORE, RUN, GH).push_now()
```

To abandon an attempt and start clean, bump `ATTEMPT`. The old store stays on the
branch untouched, so nothing is lost and the two attempts can be compared.

### 5 · Read the verdict — two scorecards, kept apart

In [6]:
s=json.load(open(f'{STORE}/run_summary.json'))
print(json.dumps(s, indent=2))
print('''
WHAT DECIDES STAGE 3

  final_accuracy > pass1_accuracy
      The pipeline improves accuracy on unseen questions with no gold in the
      loop. That is the deployable claim. Go to test at best_budget_measured,
      NOT at the budget this run happened to use.

  final ~= pass1, but AUROC >= 0.65
      Detection generalises; the retry is being spent badly. Tighten
      FLAG_BUDGET toward 0.2 so fewer correct traces get broken.

  AUROC < 0.65
      Detection did not generalise. Report it beside the FST negative.

READ THESE THREE FIRST

  auroc                  compare against 0.776, the stage-1 number on pass-1
                         traces. NOT against the 0.756 headline, which averages
                         in retry traces this stage never sees.

  measured_damage        the real cost of a false positive; 0.765 was a prior
                         from a pre-audit run. Far below it -> a looser budget
                         becomes affordable and the whole pipeline gains value.

  best_budget_measured   where the optimum actually sits once recovery and
                         damage are measured rather than assumed. This is the
                         number stage 3 uses.

AND THE GATE

  n_ungated == 0         every flagged trace fell below TYPE_GATE, so no typed
                         cure ran and this is a generic arm wearing CTC's name.
                         Nothing about type routing means anything in that case.

  recovery_ungated_typed vs recovery_gated_generic are DIFFERENT populations --
  low-confidence traces are harder. For the paper ablation, run again with
  TYPE_GATE=0 and compare final_accuracy.
''')


{
  "arm": "ctc_validate",
  "n": 400,
  "threshold": 0.683,
  "flag_rule": "--flag-budget 40% -> p_wrong >= 0.683 (FALLBACK: this model shipped no calibration table -- retrain with the D77 ctc_train2.py to calibrate on your own data)",
  "flag_budget": 0.4,
  "type_conf_gate": 0.5,
  "ctc_version": 2,
  "n_flagged": 138,
  "eval_offset": 1500,
  "model": "meta-llama/Llama-2-7b-chat-hf",
  "retry_flagged": true,
  "retrieval": "3stage",
  "pool_size": 361,
  "exemplars_from_pool_probe": 8,
  "auroc": 0.7702558635394455,
  "best_budget_prior": 0.3,
  "best_budget_measured": 0.4,
  "type_accuracy": 0.5902255639097744,
  "type_accuracy_flagged": 0.5433070866141733,
  "recovery_gated_generic": 0.2727272727272727,
  "recovery_ungated_typed": 0.18085106382978725,
  "n_gated": 33,
  "n_ungated": 94,
  "type_matters_p": 0.4035784586277541,
  "recovery_type_matched": 0.15,
  "recovery_type_mismatched": 0.23529411764705882,
  "n_type_matched": 60,
  "n_type_mismatched": 34,
  "pass1_accuracy": 0

### After this

**Stage 3** is this same script on the test split with `--no-score` and no gold
anywhere: the model solves, CTC decides, the cure loads, the model solves again.
Report pass-1 versus final accuracy on test.

Change three things and nothing else:

```python
K.SPLIT_TAG   # 'test'  -- inside run_ctc_validate.py
OFFSET = 0
'--no-score'  # added to the command line
FLAG_BUDGET = s['best_budget_measured']   # from THIS run, not the prior
```

That is the claim nothing else in the project can make. EDER needs gold to diagnose,
so it can only ever run on train. CTC is what carries the method across.